In [2]:
!pip install datasets sentence-transformers faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.9 MB/s eta 0:00:00:00:0100:01


In [4]:
from datasets import load_dataset
import pandas as pd

# This loads 1,500 rows
dataset = load_dataset("yelp_review_full", split="train[:1500]")
df = pd.DataFrame(dataset)

df = df.rename(columns={'label': 'stars'}) 

# Yelp Full dataset labels are 0-4, so let's make them 1-5 stars
df['stars'] = df['stars'] + 1 

df[['stars', 'text']].head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

,stars,text
0,5,dr. goldberg offers everything i look for in a...
1,2,"Unfortunately, the frustration of being Dr. Go..."
2,4,Been going to Dr. Goldberg for over 10 years. ...
3,4,Got a letter in the mail last week that said D...
4,1,I don't know what Dr. Goldberg was like before...


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # remove punctuations
    text = re.sub(r'\s+', ' ', text)           # removes extra space
    return text


df['cleaned_text'] = df['text'].astype(str).apply(clean_text)
df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,dr. goldberg offers everything i look for in a...,dr goldberg offers everything i look for in a ...
1,"Unfortunately, the frustration of being Dr. Go...",unfortunately the frustration of being dr gold...
2,Been going to Dr. Goldberg for over 10 years. ...,been going to dr goldberg for over 10 years i ...
3,Got a letter in the mail last week that said D...,got a letter in the mail last week that said d...
4,I don't know what Dr. Goldberg was like before...,i dont know what dr goldberg was like before m...


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np


model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


embeddings = model.encode(df['cleaned_text'].values)
embeddings = np.array(embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:

print(f"Embeddings shape: {embeddings.shape}")
print(f"Sample embedding (first 5 dimensions): {embeddings[0][:5]}")

Embeddings shape: (1500, 384)
Sample embedding (first 5 dimensions): [ 0.04380927 -0.04998112  0.05118718 -0.02572107 -0.13256662]


In [8]:
import faiss

#  Euclidean distance
dimensions = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)

# Add embeddings to the index
faiss_index.add(embeddings)

# Save the index
faiss.write_index(faiss_index, "faiss_yelp_index.index")
print(f"FAISS index saved with {faiss_index.ntotal} vectors")

FAISS index saved with 1500 vectors


In [9]:

def get_similar_reviews(query, count=5):
    query_embedding = model.encode([query])
    distance, indices = faiss_index.search(query_embedding, count)
    
    results = []
    for i in range(count):
        results.append({
            'review': df['text'].iloc[indices[0][i]],
            'stars': df['stars'].iloc[indices[0][i]],
            'distance': float(distance[0][i])
        })
    
    return results


In [10]:


test_results = get_similar_reviews("Great food and service")
for i, result in enumerate(test_results, 1):
    print(f"\n{i}. Stars: {result['stars']}, Distance: {result['distance']:.4f}")
    print(f"   {result['review'][:100]}...")


1. Stars: 5, Distance: 0.5316
   Food here is absolutely wonderful! Service is good and people are better. Take my family there and a...

2. Stars: 4, Distance: 0.6854
   Great food and very reasonable prices.   Really enjoy the sandwich and big orders of tasty fries....

3. Stars: 5, Distance: 0.7208
   I love this place! The food is always so fresh and delicious. The staff is always friendly, as well....

4. Stars: 4, Distance: 0.7251
   The service here was outstanding with a very friendly staff. I could not believe the amount of food ...

5. Stars: 4, Distance: 0.7326
   great atmosphere and great food. while the menu changes seasonally, my group all enjoyed the food - ...


In [11]:
# Save embeddings as .npy file
np.save('yelp_embeddings.npy', embeddings)
print("Embeddings saved to yelp_embeddings.npy")

Embeddings saved to yelp_embeddings.npy


In [12]:
df.to_csv('yelp_reviews.csv', index=False)
print("DataFrame saved to yelp_reviews.csv")

DataFrame saved to yelp_reviews.csv
